# Ablación de la ponderación de clases — CNN-1D multiclase

Responde al comentario 5 del Revisor 1 (JCP, manuscrito jcp-4572356).

**Qué hace:** reentrena la CNN-1D multiclase en dos configuraciones idénticas salvo por un
detalle, la presencia o ausencia de los pesos de clase inversamente proporcionales a la
frecuencia en la función de pérdida. Todo lo demás se mantiene constante: las mismas
particiones guardadas en Drive, el mismo escalador, la misma arquitectura, el mismo
optimizador, la misma semilla y el mismo criterio de parada temprana.

**Qué NO hace:** no toca ningún archivo existente. Todo lo nuevo se escribe en
`models/ablacion/`. Los pesos, historiales y resultados de las corridas anteriores
quedan intactos.

**Se puede interrumpir y reanudar.** Cada corrida guarda su resultado al terminar y las
ya completadas se saltan automáticamente.

**Tiempo estimado:** de 15 a 20 minutos para reconstruir la matriz escalada, más unos
10 a 35 minutos por cada configuración entrenada.

**Antes de empezar:** activa la GPU en Entorno de ejecución, Cambiar tipo de entorno,
Acelerador por hardware, GPU T4.

## 1. Entorno y artefactos

In [1]:
import os, gc, json, time, pickle, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/ArticuloSeguridadInformatica'
PROCESADO = f'{BASE}/data/processed'
FINAL     = f'{BASE}/data/final'
MODELOS   = f'{BASE}/models'
ABLACION  = f'{BASE}/models/ablacion'          # todo lo nuevo va aqui
os.makedirs(ABLACION, exist_ok=True)

MEMMAP_SCALED = '/content/X_all_scaled_v2.dat'
RANDOM_SEED   = 42
BLOCK         = 1_000_000
CHUNK_ROWS    = 200_000
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('GPU        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO DISPONIBLE')
print('PyTorch    :', torch.__version__)
print('Salida en  :', ABLACION)

# Metadata de la particion original
with open(f'{FINAL}/metadata.json') as f:
    META = json.load(f)
FEATURE_NAMES = META['feature_names']
N_FEATURES    = META['n_features']
CLASES        = META['clases']
TOTAL_FILAS   = META['total_filas']

assert META['random_seed'] == 42, 'La semilla de la particion no es 42'
assert N_FEATURES == 65 and TOTAL_FILAS == 11_346_511, 'Metadata inesperada'

with open(f'{FINAL}/scaler_v2.pkl', 'rb') as f:
    scaler_v2 = pickle.load(f)
with open(f'{FINAL}/log_features_idx.pkl', 'rb') as f:
    log_features_idx = pickle.load(f)

print(f'\nFeatures   : {N_FEATURES}   con log-transform: {len(log_features_idx)}')
print(f'Clases     : {len(CLASES)}')
print(f'Filas      : {TOTAL_FILAS:,}   split: {META["split"]}')

Mounted at /content/drive
GPU        : Tesla T4
PyTorch    : 2.11.0+cu128
Salida en  : /content/drive/MyDrive/ArticuloSeguridadInformatica/models/ablacion


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Features   : 65   con log-transform: 54
Clases     : 15
Filas      : 11,346,511   split: {'train': 7947096, 'val': 1697438, 'test': 1701977}


## 2. Reconstrucción de la matriz escalada

El archivo `X_all_scaled_v2.dat` vive en el almacenamiento temporal de Colab, así que hay
que regenerarlo en cada sesión. Se reconstruye en una sola pasada aplicando, por bloques,
la imputación de NaN, la transformación logarítmica firmada y el escalador robusto ya
ajustado, seguidos del recorte al rango de menos diez a diez. **No se reajusta el
escalador**, se reutiliza el guardado en Drive, de modo que los datos son idénticos a los
de la corrida original.

In [2]:
def construir_memmap_escalado():
    parquets = sorted([f for f in os.listdir(PROCESADO) if f.endswith('.parquet')])
    assert len(parquets) == 10, f'Se esperaban 10 parquet, hay {len(parquets)}'

    X = np.lib.format.open_memmap(MEMMAP_SCALED, mode='w+',
                                  dtype=np.float32, shape=(TOTAL_FILAS, N_FEATURES))
    del X; gc.collect()

    offset = 0; total_nan = 0; t_ini = time.time()
    for f in parquets:
        ruta = os.path.join(PROCESADO, f)
        df = pd.read_parquet(ruta, columns=FEATURE_NAMES)
        n  = len(df)
        X  = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r+')

        for s in range(0, n, CHUNK_ROWS):
            e = min(s + CHUNK_ROWS, n)
            blk = df.iloc[s:e].values.astype(np.float32)
            total_nan += int(np.isnan(blk).sum())
            blk = np.nan_to_num(blk, nan=0.0, posinf=0.0, neginf=0.0)
            for i in log_features_idx:
                col = blk[:, i]
                blk[:, i] = np.sign(col) * np.log1p(np.abs(col))
            blk = np.clip(scaler_v2.transform(blk), -10.0, 10.0).astype(np.float32)
            X[offset+s:offset+e] = blk

        X.flush(); offset += n
        del df, X; gc.collect()
        print(f'  {f:<45} {n:>10,} filas   ({time.time()-t_ini:>5.0f}s)')

    assert offset == TOTAL_FILAS, f'Filas escritas {offset:,} != {TOTAL_FILAS:,}'
    print(f'\n  NaN imputados a 0: {total_nan:,}')
    return total_nan

necesita = True
if os.path.exists(MEMMAP_SCALED):
    try:
        X = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r')
        necesita = X.shape != (TOTAL_FILAS, N_FEATURES)
        del X
    except Exception:
        necesita = True

if necesita:
    print('Construyendo la matriz escalada (una sola pasada)...')
    NAN_TOTAL = construir_memmap_escalado()
else:
    print('La matriz escalada ya existe en esta sesion, se reutiliza.')
    NAN_TOTAL = None

X_scaled = np.lib.format.open_memmap(MEMMAP_SCALED, mode='r')
m = X_scaled[:200_000]
print(f'\nVerificacion  shape={X_scaled.shape}  rango=[{m.min():.2f}, {m.max():.2f}]  NaN={int(np.isnan(m).sum())}')
del m; gc.collect()

Construyendo la matriz escalada (una sola pasada)...
  Friday-02-03-2018_clean.parquet                  819,904 filas   (   11s)
  Friday-16-02-2018_clean.parquet                  591,901 filas   (   21s)
  Friday-23-02-2018_clean.parquet                  905,181 filas   (   34s)
  Thuesday-20-02-2018_clean.parquet              5,435,425 filas   (   77s)
  Thursday-01-03-2018_clean.parquet                253,221 filas   (   82s)
  Thursday-15-02-2018_clean.parquet                794,613 filas   (   92s)
  Thursday-22-02-2018_clean.parquet                829,894 filas   (  104s)
  Wednesday-14-02-2018_clean.parquet               619,342 filas   (  116s)
  Wednesday-21-02-2018_clean.parquet               638,999 filas   (  123s)
  Wednesday-28-02-2018_clean.parquet               458,031 filas   (  131s)

  NaN imputados a 0: 3,270,477

Verificacion  shape=(11346511, 65)  rango=[-10.00, 10.00]  NaN=0


11

## 3. Carga de las particiones originales en memoria

In [3]:
def cargar_indices_eficiente(memmap, indices, block_size=1_000_000):
    """Lectura por bloques secuenciales, mucho mas rapida que el fancy indexing."""
    orden  = np.argsort(indices)
    ordena = indices[orden]
    out    = np.empty((len(indices), memmap.shape[1]), dtype=np.float32)
    ptr = 0
    for s in range(0, memmap.shape[0], block_size):
        e = min(s + block_size, memmap.shape[0])
        fin = ptr
        while fin < len(ordena) and ordena[fin] < e:
            fin += 1
        if fin > ptr:
            blk = memmap[s:e]
            out[orden[ptr:fin]] = blk[ordena[ptr:fin] - s]
            ptr = fin
        if ptr >= len(ordena):
            break
    return out

idx_train = np.load(f'{FINAL}/idx_train.npy')
idx_val   = np.load(f'{FINAL}/idx_val.npy')
idx_test  = np.load(f'{FINAL}/idx_test.npy')
y_train   = np.load(f'{FINAL}/y_train_multi.npy')
y_val     = np.load(f'{FINAL}/y_val_multi.npy')
y_test    = np.load(f'{FINAL}/y_test_multi.npy')

t0 = time.time(); X_train = cargar_indices_eficiente(X_scaled, idx_train); print(f'  X_train {X_train.shape}  ({time.time()-t0:.0f}s)')
t0 = time.time(); X_val   = cargar_indices_eficiente(X_scaled, idx_val);   print(f'  X_val   {X_val.shape}  ({time.time()-t0:.0f}s)')
t0 = time.time(); X_test  = cargar_indices_eficiente(X_scaled, idx_test);  print(f'  X_test  {X_test.shape}  ({time.time()-t0:.0f}s)')

assert X_train.shape[0] == META['split']['train']
assert X_val.shape[0]   == META['split']['val']
assert X_test.shape[0]  == META['split']['test']
print(f'\nRAM usada: {(X_train.nbytes+X_val.nbytes+X_test.nbytes)/1024**3:.2f} GB')

# Pesos de clase, identicos a la corrida original
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(len(CLASES)), y=y_train)
print(f'\nPesos de clase  min={cw.min():.4f} ({CLASES[int(cw.argmin())]})   max={cw.max():.2f} ({CLASES[int(cw.argmax())]})')
CW_TENSOR = torch.tensor(cw, dtype=torch.float32, device=DEVICE)

  X_train (7947096, 65)  (15s)
  X_val   (1697438, 65)  (1s)
  X_test  (1701977, 65)  (1s)

RAM usada: 2.75 GB

Pesos de clase  min=0.0761 (Benign)   max=13584.78 (DoS attacks-SlowHTTPTest)


## 4. Modelo y rutina de entrenamiento

In [4]:
class CNN1D(nn.Module):
    """Identica a la del experimento original."""
    def __init__(self, n_features=65, n_classes=15, dropout=0.3):
        super().__init__()
        self.conv1   = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.bn1     = nn.BatchNorm1d(64)
        self.conv2   = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2     = nn.BatchNorm1d(128)
        self.pool    = nn.MaxPool1d(2)
        self.fc1     = nn.Linear(128 * (n_features // 2), 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(128, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).flatten(1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)


class FastTensorLoader:
    def __init__(self, X, y=None, batch_size=512, shuffle=True, device='cuda'):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long() if y is not None else None
        self.batch_size, self.shuffle, self.device, self.n = batch_size, shuffle, device, len(X)
    def __iter__(self):
        perm = torch.randperm(self.n) if self.shuffle else torch.arange(self.n)
        for s in range(0, self.n, self.batch_size):
            i = perm[s:s+self.batch_size]
            xb = self.X[i].to(self.device, non_blocking=True)
            yield (xb, self.y[i].to(self.device, non_blocking=True)) if self.y is not None else xb
    def __len__(self):
        return (self.n + self.batch_size - 1) // self.batch_size


def entrenar(usar_pesos, semilla, etiqueta, epochs=20, patience=4):
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla); np.random.seed(semilla)
    model = CNN1D(N_FEATURES, len(CLASES), dropout=0.3).to(DEVICE)

    crit = nn.CrossEntropyLoss(weight=CW_TENSOR if usar_pesos else None)
    opt  = optim.Adam(model.parameters(), lr=1e-3)
    sch  = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

    tr = FastTensorLoader(X_train, y_train, 512,  True,  str(DEVICE))
    vl = FastTensorLoader(X_val,   y_val,   1024, False, str(DEVICE))

    hist = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    mejor, espera, ruta_pesos = 0.0, 0, f'{ABLACION}/{etiqueta}_best.pt'

    print(f'\n{"="*62}\n{etiqueta}   pesos_de_clase={usar_pesos}   semilla={semilla}\n{"="*62}')
    print(f'{"Ep":>3} {"TrainLoss":>10} {"ValLoss":>9} {"ValF1":>8} {"LR":>9} {"s":>6}')
    print('-'*52)
    for ep in range(1, epochs+1):
        t0 = time.time(); model.train(); tl = []
        for xb, yb in tr:
            opt.zero_grad(); l = crit(model(xb), yb); l.backward(); opt.step(); tl.append(l.item())
        model.eval(); vls, P, L = [], [], []
        with torch.no_grad():
            for xb, yb in vl:
                o = model(xb); vls.append(crit(o, yb).item())
                P.append(o.argmax(1).cpu().numpy()); L.append(yb.cpu().numpy())
        trl, vll = float(np.mean(tl)), float(np.mean(vls))
        f1v = f1_score(np.concatenate(L), np.concatenate(P), average='macro')
        lr = opt.param_groups[0]['lr']; sch.step(f1v)
        hist['train_loss'].append(trl); hist['val_loss'].append(vll); hist['val_f1'].append(f1v)
        marca = ''
        if f1v > mejor:
            mejor, espera = f1v, 0; torch.save(model.state_dict(), ruta_pesos); marca = '  *'
        else:
            espera += 1
        print(f'{ep:>3} {trl:>10.4f} {vll:>9.4f} {f1v:>8.4f} {lr:>9.6f} {time.time()-t0:>6.0f}{marca}')
        if espera >= patience:
            print(f'Parada temprana en la epoca {ep}'); break

    model.load_state_dict(torch.load(ruta_pesos))
    return model, hist, mejor, len(hist['val_f1'])


def evaluar(model):
    model.eval(); P = []
    ld = FastTensorLoader(X_test, None, 2048, False, str(DEVICE))
    with torch.no_grad():
        for xb in ld:
            P.append(model(xb).argmax(1).cpu().numpy())
    pred = np.concatenate(P)
    rep  = classification_report(y_test, pred, target_names=CLASES,
                                 output_dict=True, zero_division=0)
    cm   = confusion_matrix(y_test, pred)
    i_ben = CLASES.index('Benign')
    i_inf = [i for i, c in enumerate(CLASES) if c.lower().startswith('infil')][0]
    return {
        'f1_macro':     float(rep['macro avg']['f1-score']),
        'f1_weighted':  float(rep['weighted avg']['f1-score']),
        'precision_macro': float(rep['macro avg']['precision']),
        'recall_macro':    float(rep['macro avg']['recall']),
        'accuracy':     float(rep['accuracy']),
        'recall_benign': float(rep['Benign']['recall']),
        'benign_a_infiltration': int(cm[i_ben, i_inf]),
        'benign_mal_clasificados': int(cm[i_ben].sum() - cm[i_ben, i_ben]),
        'benign_total': int(cm[i_ben].sum()),
        'report_por_clase': {c: {k: float(v) for k, v in rep[c].items()} for c in CLASES},
    }

print('Modelo y funciones listos.')

Modelo y funciones listos.


## 5. Ejecución de la ablación

Las dos primeras configuraciones son las imprescindibles para responder al revisor. Las
cuatro restantes añaden dos semillas más por brazo y sirven para medir la variabilidad
entre ejecuciones, que es el otro punto que los revisores cuestionan. Puedes dejarlas
comentadas y ejecutarlas después: **las corridas ya completadas se saltan solas**.

In [8]:
CONFIGS = [
    ('multiclase_con_pesos_s42',  True,  42),
    ('multiclase_sin_pesos_s42',  False, 42),
    # Descomenta para la medicion de variabilidad entre semillas:
    ('multiclase_con_pesos_s7',   True,   7),
    ('multiclase_sin_pesos_s7',   False,  7),
    ('multiclase_con_pesos_s123', True, 123),
    ('multiclase_sin_pesos_s123', False, 123),
]

for etiqueta, usar_pesos, semilla in CONFIGS:
    destino = f'{ABLACION}/{etiqueta}_resultados.json'
    if os.path.exists(destino):
        print(f'[ya existe, se salta] {etiqueta}')
        continue

    t0 = time.time()
    model, hist, mejor_val, n_ep = entrenar(usar_pesos, semilla, etiqueta)
    met = evaluar(model)

    salida = {
        'etiqueta': etiqueta, 'pesos_de_clase': usar_pesos, 'semilla': semilla,
        'epocas_ejecutadas': n_ep, 'mejor_val_f1_macro': float(mejor_val),
        'minutos': round((time.time()-t0)/60, 1),
        'fecha': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'torch': torch.__version__,
        'test': met, 'history': hist,
    }
    with open(destino, 'w') as f:
        json.dump(salida, f, indent=2, ensure_ascii=False)

    print(f'\n  F1-macro test      : {met["f1_macro"]:.4f}')
    print(f'  F1-ponderado test  : {met["f1_weighted"]:.4f}')
    print(f'  Exactitud global   : {met["accuracy"]:.4f}')
    print(f'  Recall de Benign   : {met["recall_benign"]:.4f}')
    print(f'  Benign -> Infiltration: {met["benign_a_infiltration"]:,} '
          f'({100*met["benign_a_infiltration"]/met["benign_total"]:.1f}%)')
    print(f'  Guardado en {destino}')
    del model; gc.collect(); torch.cuda.empty_cache()

print('\nAblacion terminada.')

[ya existe, se salta] multiclase_con_pesos_s42
[ya existe, se salta] multiclase_sin_pesos_s42

multiclase_con_pesos_s7   pesos_de_clase=True   semilla=7
 Ep  TrainLoss   ValLoss    ValF1        LR      s
----------------------------------------------------
  1     0.2683    0.1840   0.5784  0.001000    110  *
  2     0.1799    0.1680   0.5728  0.001000    108
  3     0.1661    0.1626   0.5741  0.001000    107
  4     0.1596    0.1729   0.6021  0.001000    106  *
  5     0.1594    0.1697   0.5812  0.001000    106
  6     0.1570    0.1528   0.5832  0.001000    106
  7     0.1545    0.1605   0.5772  0.001000    105
  8     0.1474    0.1523   0.5792  0.000500    105
Parada temprana en la epoca 8

  F1-macro test      : 0.5946
  F1-ponderado test  : 0.6901
  Exactitud global   : 0.5543
  Recall de Benign   : 0.4925
  Benign -> Infiltration: 748,584 (50.2%)
  Guardado en /content/drive/MyDrive/ArticuloSeguridadInformatica/models/ablacion/multiclase_con_pesos_s7_resultados.json

multiclase_si

## 6. Tabla comparativa y resumen

In [6]:
import glob
res = []
for p in sorted(glob.glob(f'{ABLACION}/*_resultados.json')):
    res.append(json.load(open(p)))
assert res, 'No hay resultados todavia.'

print('='*104)
print('ABLACION DE LA PONDERACION DE CLASES — CNN-1D MULTICLASE (test n = 1,701,977)')
print('='*104)
print(f'{"Configuracion":<28}{"Pesos":>7}{"Sem":>5}{"Ep":>4}{"F1-macro":>10}{"F1-pond":>10}'
      f'{"Exact.":>9}{"Rec.Benign":>12}{"Ben->Infil":>13}')
print('-'*104)
for r in res:
    t = r['test']
    print(f'{r["etiqueta"]:<28}{str(r["pesos_de_clase"]):>7}{r["semilla"]:>5}{r["epocas_ejecutadas"]:>4}'
          f'{t["f1_macro"]:>10.4f}{t["f1_weighted"]:>10.4f}{t["accuracy"]:>9.4f}'
          f'{t["recall_benign"]:>12.4f}{100*t["benign_a_infiltration"]/t["benign_total"]:>12.1f}%')
print('='*104)

# Efecto de los pesos, comparando por semilla
print('\nEFECTO DE LA PONDERACION (misma semilla, unica diferencia los pesos)')
print('-'*104)
for s in sorted({r['semilla'] for r in res}):
    con = next((r for r in res if r['semilla']==s and r['pesos_de_clase']), None)
    sin = next((r for r in res if r['semilla']==s and not r['pesos_de_clase']), None)
    if con and sin:
        d = con['test']['f1_macro'] - sin['test']['f1_macro']
        print(f'  semilla {s:>3}:  con pesos {con["test"]["f1_macro"]:.4f}   '
              f'sin pesos {sin["test"]["f1_macro"]:.4f}   diferencia {d:+.4f}')

# Clases minoritarias, que es donde la ponderacion deberia notarse
MINORITARIAS = ['DoS attacks-SlowHTTPTest','SQL Injection','FTP-BruteForce',
                'Brute Force -XSS','Brute Force -Web','DDOS attack-LOIC-UDP']
print('\nF1 POR CLASE MINORITARIA')
print('-'*104)
print(f'{"Clase":<30}' + ''.join(f'{r["etiqueta"][-16:]:>20}' for r in res))
print('-'*104)
for c in MINORITARIAS:
    fila = f'{c:<30}'
    for r in res:
        v = r['test']['report_por_clase'].get(c, {}).get('f1-score')
        fila += f'{v:>20.4f}' if v is not None else f'{"-":>20}'
    print(fila)

# Variabilidad entre semillas
for pesos in [True, False]:
    vals = [r['test']['f1_macro'] for r in res if r['pesos_de_clase']==pesos]
    if len(vals) > 1:
        print(f'\nVariabilidad entre semillas (pesos={pesos}): '
              f'media {np.mean(vals):.4f}  desv.est. {np.std(vals, ddof=1):.4f}  '
              f'rango [{min(vals):.4f}, {max(vals):.4f}]  n={len(vals)}')

resumen = {'fecha': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
           'n_configuraciones': len(res),
           'corridas': [{'etiqueta': r['etiqueta'], 'pesos_de_clase': r['pesos_de_clase'],
                         'semilla': r['semilla'], 'epocas': r['epocas_ejecutadas'],
                         **{k: r['test'][k] for k in
                            ['f1_macro','f1_weighted','accuracy','recall_benign',
                             'benign_a_infiltration','benign_mal_clasificados']}}
                        for r in res]}
with open(f'{ABLACION}/RESUMEN_ABLACION.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(f'\nResumen guardado en {ABLACION}/RESUMEN_ABLACION.json')

ABLACION DE LA PONDERACION DE CLASES — CNN-1D MULTICLASE (test n = 1,701,977)
Configuracion                 Pesos  Sem  Ep  F1-macro   F1-pond   Exact.  Rec.Benign   Ben->Infil
--------------------------------------------------------------------------------------------------------
multiclase_con_pesos_s42       True   42  12    0.6026    0.7852   0.6690      0.6248        37.0%
multiclase_sin_pesos_s42      False   42  18    0.7521    0.9842   0.9888      0.9993         0.0%

EFECTO DE LA PONDERACION (misma semilla, unica diferencia los pesos)
--------------------------------------------------------------------------------------------------------
  semilla  42:  con pesos 0.6026   sin pesos 0.7521   diferencia -0.1495

F1 POR CLASE MINORITARIA
--------------------------------------------------------------------------------------------------------
Clase                             se_con_pesos_s42    se_sin_pesos_s42
----------------------------------------------------------------------

In [7]:
import json, glob, os
ABLACION = '/content/drive/MyDrive/ArticuloSeguridadInformatica/models/ablacion'
res = {os.path.basename(p): json.load(open(p)) for p in sorted(glob.glob(f'{ABLACION}/*_resultados.json'))}
con = next(r for r in res.values() if r['pesos_de_clase'])
sin = next(r for r in res.values() if not r['pesos_de_clase'])

print('='*108)
print('COMPARACION POR CLASE — 15 clases, conjunto de prueba')
print('='*108)
print(f'{"Clase":<28}{"Sop.":>9} | {"P con":>7}{"R con":>7}{"F1 con":>8} | {"P sin":>7}{"R sin":>7}{"F1 sin":>8} | {"dF1":>8}')
print('-'*108)
for c in con['test']['report_por_clase']:
    a, b = con['test']['report_por_clase'][c], sin['test']['report_por_clase'][c]
    print(f'{c:<28}{int(a["support"]):>9,} | {a["precision"]:>7.4f}{a["recall"]:>7.4f}{a["f1-score"]:>8.4f} | '
          f'{b["precision"]:>7.4f}{b["recall"]:>7.4f}{b["f1-score"]:>8.4f} | {b["f1-score"]-a["f1-score"]:>+8.4f}')
print('='*108)

for nom, r in [('CON pesos', con), ('SIN pesos', sin)]:
    t = r['test']
    print(f'\n{nom}:  epocas={r["epocas_ejecutadas"]}  mejor_val_F1={r["mejor_val_f1_macro"]:.4f}  minutos={r["minutos"]}')
    print(f'   Benign mal clasificados: {t["benign_mal_clasificados"]:,} de {t["benign_total"]:,} '
          f'({100*t["benign_mal_clasificados"]/t["benign_total"]:.2f}%)   a Infiltration: {t["benign_a_infiltration"]:,}')
    print(f'   val_f1 por epoca: {[round(x,4) for x in r["history"]["val_f1"]]}')

COMPARACION POR CLASE — 15 clases, conjunto de prueba
Clase                            Sop. |   P con  R con  F1 con |   P sin  R sin  F1 sin |      dF1
------------------------------------------------------------------------------------------------------------
Benign                      1,490,671 |  0.9963 0.6248  0.7680 |  0.9882 0.9993  0.9937 |  +0.2257
Bot                            21,680 |  0.9770 0.9914  0.9842 |  0.9994 0.9955  0.9974 |  +0.0133
Brute Force -Web                   86 |  0.0286 0.9186  0.0555 |  0.9524 0.4651  0.6250 |  +0.5695
Brute Force -XSS                   34 |  0.1076 0.7059  0.1868 |  1.0000 0.7059  0.8276 |  +0.6408
DDOS attack-HOIC               41,470 |  0.9592 0.9994  0.9789 |  0.9961 0.9998  0.9979 |  +0.0190
DDOS attack-LOIC-UDP              260 |  0.7143 1.0000  0.8333 |  0.8614 0.8846  0.8729 |  +0.0395
DDoS attacks-LOIC-HTTP         86,366 |  0.9827 0.9982  0.9904 |  0.9984 0.9985  0.9985 |  +0.0081
DoS attacks-GoldenEye           6,211 |  0.98

In [9]:
import json, glob, numpy as np
ABL = '/content/drive/MyDrive/ArticuloSeguridadInformatica/models/ablacion'
R   = [json.load(open(p)) for p in sorted(glob.glob(f'{ABL}/*_resultados.json'))]
CON = [r for r in R if r['pesos_de_clase']]
SIN = [r for r in R if not r['pesos_de_clase']]
print(f'Corridas encontradas: {len(CON)} con pesos, {len(SIN)} sin pesos\n')

def ag(grupo, clase, metrica):
    v = np.array([r['test']['report_por_clase'][clase][metrica] for r in grupo], dtype=float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0)

clases = list(CON[0]['test']['report_por_clase'])

print('='*104)
print('F1 POR CLASE  (media ± desviación estándar sobre las semillas)')
print('='*104)
print(f'{"Clase":<28}{"Soporte":>9}{"CON pesos":>21}{"SIN pesos":>21}{"ΔF1":>11}')
print('-'*104)
for c in clases:
    sop = int(CON[0]['test']['report_por_clase'][c]['support'])
    a, sa = ag(CON, c, 'f1-score'); b, sb = ag(SIN, c, 'f1-score')
    print(f'{c:<28}{sop:>9,}{a:>13.4f} ± {sa:.4f}{b:>13.4f} ± {sb:.4f}{b-a:>+11.4f}')

print('\n' + '='*104)
print('PRECISION Y RECALL POR CLASE  (media sobre las semillas)')
print('='*104)
print(f'{"Clase":<28}{"P con":>11}{"R con":>11}{"P sin":>11}{"R sin":>11}{"ΔP":>11}{"ΔR":>11}')
print('-'*104)
for c in clases:
    pa,_ = ag(CON, c, 'precision'); ra,_ = ag(CON, c, 'recall')
    pb,_ = ag(SIN, c, 'precision'); rb,_ = ag(SIN, c, 'recall')
    print(f'{c:<28}{pa:>11.4f}{ra:>11.4f}{pb:>11.4f}{rb:>11.4f}{pb-pa:>+11.4f}{rb-ra:>+11.4f}')
print('='*104)

Corridas encontradas: 3 con pesos, 3 sin pesos

F1 POR CLASE  (media ± desviación estándar sobre las semillas)
Clase                         Soporte            CON pesos            SIN pesos        ΔF1
--------------------------------------------------------------------------------------------------------
Benign                      1,490,671       0.7205 ± 0.0555       0.9936 ± 0.0000    +0.2731
Bot                            21,680       0.9799 ± 0.0058       0.9972 ± 0.0004    +0.0173
Brute Force -Web                   86       0.0576 ± 0.0089       0.6202 ± 0.0083    +0.5626
Brute Force -XSS                   34       0.2093 ± 0.0215       0.8276 ± 0.0000    +0.6183
DDOS attack-HOIC               41,470       0.9862 ± 0.0064       0.9970 ± 0.0012    +0.0107
DDOS attack-LOIC-UDP              260       0.8338 ± 0.0008       0.8590 ± 0.0212    +0.0253
DDoS attacks-LOIC-HTTP         86,366       0.9910 ± 0.0008       0.9982 ± 0.0002    +0.0072
DoS attacks-GoldenEye           6,211     